## Get number of s3 objects

Let us go through the details about how we can get number of s3 objects. We will understand the relevance of **Marker** to paginate `list_objects` output using boto3.

* One of the way to get s3 object metadata from a given bucket is to use `list_objects`.
* However, `list_objects` gets metadata only for 1000 objects at max.
* We need to paginate using `Marker` and iterate until we get details about all the objects.

Here are the steps we can follow to get the number of s3 objects with in a bucket.
* Create s3 client with appropriate profile.
* Invoke list_objects incrementally using `Marker` until you get details about all the objects.
* Get number of elements in the `Contents` and add it to object count. We can break the loop when the size of `Contents` list is less than 1000 or when `Contents` does not exists as part of the response.

In [1]:
import boto3
import os
os.environ.setdefault('http_proxy', 'http://webproxy...com:')
os.environ.setdefault('https_proxy', 'http://webproxy...com:')
os.environ.setdefault('AWS_PROFILE', 'itvgenlogs')
BUCKET_NAME = 'itv-genlogs'
PREFIX = 'logs/year'

In [33]:
s3_client = boto3.client('s3')

In [35]:
marker = ''                              # initialize to empty marker
object_count = 0
MAX_KEYS = 50
while True:
    s3_objects = s3_client \
        .list_objects(                   # get objects in the bucket and prefix
            Bucket=BUCKET_NAME, 
            Prefix=PREFIX, 
            Marker=marker,
            MaxKeys=MAX_KEYS             # max keys returned for each loop
        ) \
        .get('Contents')
    if not s3_objects:
        break
    object_count += len(s3_objects)
    marker = s3_objects[-1]['Key']       # get next batch after this marker (pagination)
    print(marker)

logs/year=2026/month=01/day=07/gen_logs_s3-3-2026-01-07-04-06-53-23c10054-b19e-42df-b415-6332c010b31a
logs/year=2026/month=01/day=07/gen_logs_s3-3-2026-01-07-04-59-45-454a5c1e-7f26-4f03-90bd-6c4fe06aec2a
logs/year=2026/month=01/day=07/gen_logs_s3-3-2026-01-07-05-45-30-74496633-722f-4beb-96c1-d3107610dd4d


In [36]:
object_count

145

In [37]:
s3_objects = s3_client.list_objects(
    Bucket=BUCKET_NAME, 
    Prefix=PREFIX
)

In [17]:
a = s3_objects.get('Markers')

In [19]:
print(a)

None


In [15]:
s3_objects['Markers']

KeyError: 'Markers'

In [10]:
s3_objects['Marker']

''

In [6]:
s3_objects['MaxKeys']

1000

In [5]:
len(s3_objects['Contents'])

120

In [7]:
marker = s3_objects['Contents'][-1]['Key']

In [8]:
marker

'logs/year=2026/month=01/day=07/gen_logs_s3-3-2026-01-07-05-20-05-00bd5768-bdea-458e-a06e-95ea86f5155e'

In [11]:
s3_objects = s3_client.list_objects(
    Bucket=BUCKET_NAME, 
    Prefix=PREFIX,
    Marker=marker
)

In [12]:
s3_objects['Marker']

'logs/year=2026/month=01/day=07/gen_logs_s3-3-2026-01-07-05-20-05-00bd5768-bdea-458e-a06e-95ea86f5155e'

In [14]:
len(s3_objects['Contents'])

6